In [ ]:
# Link Dataset
# https://github.com/IndoNLP/indonlu/tree/master/dataset/smsa_doc-sentiment-prosa

# Referenace
# https://github.com/crypter70/Sentiment-Analysis-with-IndoBERT-Fine-tuning-and-IndoNLU-SmSA-Dataset/blob/main/Sentiment%20Analysis%20with%20IndoBERT%20Fine-tuning%20and%20IndoNLU%20SmSA%20Dataset.ipynb

#Sentimen Analysis (IndoBERT) FIne-tuning and IndoNLU Dataset

## Instalasi

In [ ]:
!pip install evaluate

## Import

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
from sklearn import metrics

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import evaluate
import torch
import os
import pandas as pd
import random
from IPython.display import display, HTML

In [ ]:
# Cek Device: Prioritas CUDA (GPU Colab) -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU tidak ditemukan. Menggunakan CPU (Proses training akan lebih lambat).")

## Ambil Dataset dari Github

In [ ]:
# Link dataset
url_train = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
url_valid = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"
url_test  = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/test_preprocess.tsv"

df_train = pd.read_csv(url_train, sep='\t', header=None, names=['text', 'label'])
df_valid = pd.read_csv(url_valid, sep='\t', header=None, names=['text', 'label'])
df_test  = pd.read_csv(url_test, sep='\t', header=None, names=['text', 'label'])

# Bungkus ke DatasetDict agar variabel 'dataset' siap dipakai sel-sel berikutnya
dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'validation': Dataset.from_pandas(df_valid),
    'test': Dataset.from_pandas(df_test)
})

print("Dataset Siap!")

In [ ]:
display(dataset)

In [ ]:
dataset["train"][0]

In [ ]:
# --- CONFIGURATION ---
MODEL_CHECKPOINT = "indobenchmark/indobert-base-p1"
MODEL_DIR = "IndoBERT-Sentiment-Analysis"

LEARNING_RATE = 2e-5
BATCH_SIZE = 16

EPOCHS = 5

## Preprocessing

In [ ]:
# 1. Buat mapping manual

mapping = {'positive': 0, 'neutral': 1, 'negative': 2}

# 2. Ubah kolom label di setiap dataframe
df_train['label'] = df_train['label'].map(mapping)
df_valid['label'] = df_valid['label'].map(mapping)
df_test['label'] = df_test['label'].map(mapping)

# 3. masukkan ke DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'validation': Dataset.from_pandas(df_valid),
    'test': Dataset.from_pandas(df_test)
})

In [ ]:
dataset["train"][0]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [ ]:
tokenized = dataset.map(preprocess_function, batched=True)
tokenized

##Setup Model

In [ ]:
# --- SETUP MODEL ---
id2label = {0: "POSITIVE", 1: "NEUTRAL", 2: "NEGATIVE"}
label2id = {"POSITIVE": 0, "NEUTRAL": 1, "NEGATIVE": 2}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=3, id2label=id2label, label2id=label2id)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = metrics.accuracy_score(labels, predictions)
    f1_score = metrics.f1_score(labels, predictions, average='weighted')

    return {"accuracy": accuracy, "f1_score": f1_score}

## Fine-tuning

In [ ]:
# Fine - Tuning

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy="steps",
    logging_steps=10,
    eval_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

##Evaluasi

In [ ]:
trainer.evaluate()

##Simpan Model

In [ ]:
trainer.save_model(MODEL_DIR)

##Menghapus Checkpoin yang tidak berguna

In [ ]:
import shutil
import os

if os.path.exists('model_porto.zip'):
    os.remove('model_porto.zip')
    print("File zip besar sudah dihapus.")

# Hapus semua folder checkpoint
!rm -rf IndoBERT-Sentiment-Analysis/checkpoint-*
print("Folder checkpoint sudah dibersihkan")

## Mengubah Model menjadi bentuk Zip Agar mudah untuk disimpan

In [ ]:
!zip -r model_porto.zip IndoBERT-Sentiment-Analysis/

##Pindah Model zip ke Folder yang diinginkan

In [ ]:
# Copy file dari folder lokal Colab ke folder baru di Drive
!cp /content/model_porto.zip "/content/drive/MyDrive/model-indobert/"

print("File berhasil dipindahkan ke Drive!")

In [ ]:
predictions = trainer.predict(tokenized["test"])
predicted_labels = predictions.predictions.argmax(-1)
true_labels = tokenized["test"]["label"]

accuracy = accuracy_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels, average='weighted')

print("Accuracy:", accuracy)
print("F1-score:", f1)

# Classification report
print("Classification Report:")
print(classification_report(true_labels, predicted_labels))

##Testing

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
cm

In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Confusion Matrix", y=1.05)

g = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Positive", "Neutral", "Negative"],
            yticklabels=["Positive", "Neutral", "Negative"])

g.set(ylabel='True Label', xlabel='Predicted Label')
plt.show()

In [ ]:
# Using Model

##Cara Pakai Model

###Unzip folder dari model indobert

In [ ]:
!unzip -o "/content/drive/MyDrive/model-indobert/model_porto.zip" -d /content/temp_model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Masuk path folder model
model_path = "/content/temp_model"

# Load Model & Tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=3)
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

# 3. Setup Pipeline dengan Truncation (untuk teks panjang)
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512
)

In [ ]:
#Uji COba

In [ ]:
teks_berita = "kepolisian daerah polda metro jaya menutup tahun  dengan sejumlah pencapaian kinerja positif sepanjang  polda metro jaya telah menuntaskan sejumlah tindak pidana hingga memelihara stabilitas keamanan ibu kota tetap terjaga selain melakukan penegakan hukum polda metro jaya juga berkomitmen untuk terus mendukung asta cita presiden prabowo subianto dalam menyongsong indonesia emas  mulai dari program ketahanan pangan hingga pemenuhan gizi melalui program makan bergizi gratis mbg di tengah dinamika sosial dan politik di ibu kota polda metro jaya sepanjang  ini menerima setidaknya  laporan polisi lp tertinggi seindonesia dengan demikian polda metro jaya berkontribusi sekitar  persen dari total  laporan polisi di tingkat nasional scroll to continue with content sepanjang  polda metro jaya termasuk salah satu polda dengan jumlah laporan polisi tertinggi di indonesia berkontribusi sekitar  persen dari total laporan polisi nasional kata kapolda metro jayam irjen asep edi suheri dalam konferensi pers akhir tahun di mapolda metro jaya jakarta rabu  meski demikian kapolda menyampaikan angka kriminalitas di jakarta dan sekitarnya mengalami penurunan di tahun  ini oleh karena itu irjen asep menyampaikan apresiasi kepada seluruh jajaran personel yang telah bekerja keras sepanjang  ini sepanjang tahun  ini saudarasaudara sekalian telah bekerja dalam ritme metropolitan yang tidak pernah benarbenar berhenti mulai dari mengamankan kegiatan masyarakat menangani perkara mengurai kemacetan merespons panggilan bantuan dan hadir pada saat warga membutuhkan ujar irjen asep edi kapolda memuji seluruh personel yang telah bekerja keras menguras waktu dan pikiran namun tetap menjalankan tugastugasnya dengan penuh rasa tanggung jawab semoga seluruh pengabdian tersebut benarbenar bermanfaat bagi masyarakat bangsa dan negara terutama dalam mendukung visi indonesia emas tahun  ucapnya ucapan terima kasih juga tak lupa disampaikan oleh kapolda kepada jajaran forkopimda dan seluruh elemen masyarakat yang telah berperan aktif dalam menjaga situasi kamtibmas di jakarta dan sekitarnya tetap terjaga berikut sejumlah capaian kinerja polda metro jaya sepanjang  yang dipaparkan kapolda metro irjen asep edi suheri yang detikcom rangkum kamis  selama periode januaridesember  polda metro jaya mencatatkan pengamanan berbagai kegiatan berskala besar termasuk  kali unjuk rasa di wilayah jakarta dan sekitarnya prinsip utama kami adalah kami akan berupaya menjaga keseimbangan antara pemenuhan hak warga untuk menyampaikan aspirasi tetap dihormati namun ketertiban umum keselamatan dan aktivitas masyarakat lainnya juga harus tetap terjaga ujar irjen asep dalam upaya perang melawan narkoba polda metro jaya sepanjang  telah menangkap ribuan tersangka kapolda menegaskan penegakan hukum dari hulu ke hilir akan terus dilakukan untuk mencegah penyalahgunaan narkoba untuk narkotika kami menangani  perkara dengan jumlah tersangka sebanyak  orang cukup tinggi sebagai penegasan hingga menjelang akhir tahun penindakan narkotika tetap kami lakukan katanya kapolda menyampaikan salah satu pengungkapan kasus narkoba yang paling menonjol adalah sabu seberat  kilogram dari jaringan asia tenggara oleh polres metro jakarta pusat pada kesempatan yang sama direktur narkoba polda metro jaya kombes ahmad david menjelaskan sepanjang  pihaknya dan jajaran polres telah menyita berbagai jenis narkoba seberat  ton keseluruhan barang bukti yang diamankan atau disita oleh direktorat narkoba polda metro jaya dan polres jajaran apabila kita konversi dengan nilai jual yang ada di peredaran gelap narkoba maka polda metro jaya telah mengamankan atau menyita sebesar rp  triliun dan telah menyelamatkan sebanyak  jiwa manusia kata ahmad david polda metro jaya mengungkap  kasus terkait premanisme sepanjang tahun  sebanyak  tersangka ikut ditangkap sepanjang tahun  terdapat  kasus dengan  tersangka di antaranya ada dua kejadian menonjol pertama pendudukan lahan parkir rsud tangsel dan pemerasan pedagang di pasar sgc kata dirkrimum polda metro jaya kombes iman imannudin iman menyampaikan penangkapan premanpreman ini berdampak positif terhadap pertumbuhan iklim investasi di jakarta direktorat reserse kriminal umum menindak tindak pidana perdagangan orang tppo dan tindak pidana terhadap perempuan dan anak berdasarkan data yang ada jumlah kejahatan terhadap kelompok rentan pada  turun  persen dibanding pada  sepanjang tahun  pengungkapan kasus tppo dan perlindungan perempuan dan anak ada  kasus untuk tppo dan  kasus untuk ppa di mana sudah ada  tersangka tppo  tindak pidana ppa tuturnya salah satu kasus yang menonjol adalah perdagangan anak ke salah satu suku yang ada di indonesia polda metro jaya berhasil mengembalikan anak yang dijual tersebut kepada keluarganya beberapa kasus menonjol kami informasikan kami ambil tiga contoh kasus menonjol pertama eksploitasi anak di jakbar di mana korban diimingi pekerjaan namun pelaksanaannya korban dipekerjakan dan dieksploitasi secara seksual jelasnya dalam bidang lalu lintas polda metro jaya berkomitmen untuk terus melakukan peningkatan dalam upaya mencegah transaksional di lapangan upaya ini diwujudkan dengan moderinisasi penegakan hukum lalu lintas melalui electronictraffic law enforcement etle dari aspek lalu lintas kami melihat adanya pergeseran pola penegakan hukum yang semakin modern hal ini ditandai dengan meningkatnya pelanggaran yang termonitor melalui etle dibandingkan tahun  kata irjen asep kapolda menambahkan penindakan tersebut bukan sekadar angka tetapi bagian dalam upaya membangun budaya tertib lalu lintas sementara itu polda metro jaya mencatat terdapat  kejadian kecelakaan lalu lintas dengan korban meninggal dunia sebanyak  orang dan korban lukaluka sebanyak  orang polda metro jaya terus mengoptimalkan pelayanan publik melalui layanan darurat  sepanjang  ini polda metro jaya menerima ratusan ribu panggilan bantuan membuktikan bahwa polri semakin dipercaya oleh masyarakat sepanjang tahun  kami menerima lebih dari  ribu panggilan yang masuk dan sekitar  persen panggilan terlayani oleh jajaran kami bagi kami angka ini menunjukkan bahwa masyarakat semakin percaya untuk melapor dan meminta bantuan kata kapolda metro jaya irjen asep edi suheri dalam rilis akhir tahun di mapolda metro jaya jakarta rabu  kapolda menyampaikan ukuran kepuasan masyarakat sangat ditentukan oleh seberapa cepat kehadiran anggota polri di tengah publik oleh karena itu polda metro jaya berkomitmen untuk terus mengoptimalkan layanan publik melalui hotline  dirlantas polda metro jaya komarudin menyebut angka kecelakaan lalu lintas di tahun  masih tinggi salah satu penyebabnya lantaran masih banyaknya pelanggaran lalu lintas yang dilakukan para pengendara ribuan kamera juga tersebar di jakarta untuk memantau situasi arus lalu lintas kamera tersebut juga berfungsi sebagai respon cepat polantas terkait peristiwa yang terjadi dalam hal ini diwujudkan dalam program mandala quick respons pemanfaatan mandala quick respons terbukti mampu mengurai kemacetan kita akan memantau ruasruas jalan yang memang saat itu terjadi kepadatan sehingga kami bisa dengan cepat menggeser personilpersonil kami kepada titiktitik yang memang membutuhkan penanganan segera imbuhnya mandala quick response adalah program kolaborasi polda metro jaya dan pemprov dki jakarta untuk pemantauan lalu lintas realtime menggunakan ribuan cctv bertujuan mempercepat respons darurat ambulans derek dan penanganan kemacetan dengan mengoptimalkan jalur melalui posko kendali terpadu dan operator gabungan polri dishub satpol pp untuk menciptakan rute bebas hambatan komarudin menambahkan program mandala quick respons juga berdampak pada penguraian kepadatan arus lalu lintas dia menyebut program ini mampu mengurai dan menurunkan kemacetan di jakarta pada tahun  dengan aplikasi ini kita menggeser personel ke titiktitik yang membutuhkan penanganan kita bisa satu jam lebih cepat mengembalikan masyarakat ke alamat masingmasing ke rumah masingmasing sehingga jakarta bisa kita urai di pukul  sampai dengan  pungkasnya lihat juga video polda metro terima  ribu laporan di  tertinggi seindonesia gambasvideo detik"

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")

In [ ]:
teks_berita = "ekspor minyak venezuela jatuh ke titik terendah usai amerika serikat as memblokade seluruh kapal tanker negara tersebut langkah ini diumumkan usai militer as mengklaim berhasil menangkap presiden venezuela nicolas maduro beserta istrinya di caracas berdasarkan sumber laporan reuters saat ini pengiriman minyak venezuela lumpuh karena tak kunjung mendapat izin berlayar sejak hari sabtu  kondisi ini terjadi menyusul pengumuman presiden as donald trump yang mengaku akan mengawasi transisi politik di venezuela dalam pengumuman tersebut trump mengaku mengembargo minyak dari negara amerika selatan tersebut imbasnya terdapat beberapa kapal minyak mentah dan bahan bakar tujuan as dan asia gagal berlayar dari pelabuhan minyak utama venezuela scroll to continue with content kondisi ini juga disebut akan memangkas produksi minyak venezuela lebih cepat pasalnya kapasitas tangki penyimpanan minyak di negara tersebut nyaris terisi penuh dalam beberapa pekan terakhir berdasarkan sumber dan dokumen perusahaan bumn venezuela petroleos de venezuela pdvsa selain itu embargo total minyak ini juga berdampak pada operasional perusahaan minyak asal as chevron namun hingga berita ini dimuat pdvsa dan chevron disebut belum menanggapi permintaan komentar sebagai informasi nicolas maduro dikabarkan tiba di pangkalan militer as setelah ditangkap di caracas venezuela maduro dikawal ketat agen fbi saat menginjakkan kaki di as dilansir afp minggu  maduro terlihat dikelilingi oleh agen fbi saat ia menuruni tangga pesawat pemerintah as di fasilitas garda nasional negara bagian new york dan perlahanlahan dikawal di sepanjang landasan pemimpin sayap kiri venezuela itu diperkirakan akan diterbangkan dengan helikopter ke kota new york tempat ia menghadapi tuduhan perdagangan narkoba"

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")

In [ ]:
teks_berita = "rentetan ledakan dilaporkan terdengar di area caracas ibu kota venezuela ledakan itu rupanya rentetan serangan dari amerika serikat as dilansir detiknews rentetan serangan itu diumumkan oleh presiden as donald trump trump mengklaim negaranya secara sukses melancarkan serangan skala besar terhadap venezuela pada sabtu  dini hari amerika serikat telah berhasil melancarkan serangan skala besar terhadap venezuela ucap trump dalam pernyataan via media sosial truth social seperti dilansir afp sabtu  scroll to continue with content pernyataan ini mengonfirmasi tuduhan presiden venezuela nicolas maduro yang menyebut serangan itu datang dari as washington disebut melancarkan rentetan serangan terhadap instalasi militer dan sipil di beberapa wilayah venezuela dalam pernyataannya trump juga menyebut maduro dan istrinya berhasil ditangkap dan telah diterbangkan ke luar venezuela trump tidak menjelaskan lebih lanjut soal penangkapan maduro tersebut dia hanya menyatakan bahwa operasi militer di venezuela ini dilaksanakan bekerja sama dengan otoritas penegak hukum as disebutkan trump bahwa konferensi pers akan digelar pada sabtu  siang waktu as di kediamannya maralago di florida detail lebih lanjut akan menyusul sebut trump setidaknya tujuh ledakan dan suara pesawat terbang rendah terdengar di ibu kota venezuela dengan sejumlah video menunjukkan kepulan asap menjulang ke udara dari beberapa lokasi yang berbeda presiden venezuela nicolas maduro pun menetapkan keadaan darurat pemerintahan maduro dalam pernyataannya seperti dilansir afp dan cnn sabtu  menyampaikan kecaman keras untuk agresi militer yang sangat serius dan berat oleh as terhadap venezuela venezuela menolak menyangkal dan mengecam di hadapan komunitas internasional agresi militer yang sangat serius yang dilakukan oleh pemerintah amerika serikat saat ini terhadap wilayah dan rakyat venezuela demikian pernyataan pemerintahan maduro pemerintahan maduro dalam pernyataannya menuduh as melancarkan serangan terhadap wilayah caracas dan negara bagian miranda aragua serta la guaira maduro menurut pernyataan pemerintah venezuela telah menandatangani penetapan keadaan darurat eksternal dan memerintahkan semua rencana pertahanan nasional untuk diimplementasikan pada waktu yang tepat dan dalam keadaan yang tepat pemerintah venezuela juga menyerukan para pendukungnya untuk turun ke jalanan demi membela negara rakyat turun ke jalan demikian bunyi pernyataan pemerintah venezuela menyusul rentetan ledakan terdengar di area ibu kota caracas pemerintahan bolivarian menyerukan kepada semua kekuatan sosial dan politik di negara ini untuk mengaktifkan rencana mobilisasi dan menolak serangan imperialis ini tegas pernyataan tersebut rakyat venezuela dan angkatan bersenjata nasional bolivarian dalam kesatuan militerpolisirakyat yang sempurna dikerahkan untuk menjamin kedaulatan dan perdamaian imbuh pernyataan pemerintah venezuela tersebut pernyataan itu juga menyebutkan bahwa venezuela akan menyampaikan aduan kepada dewan keamanan perserikatan bangsabangsa pbb sekretaris jenderal pbb dan badanbadan internasional lainnya untuk menuntut kecaman terhadap as saat mengumumkan serangan skala besar as terhadap venezuela trump juga mengatakan bahwa presiden nicolas maduro telah ditangkap bersama istrinya dan diterbangkan ke luar negeri trump dalam pernyataan via media sosial truth social seperti dilansir afp sabtu  mengakui bahwa as memang melancarkan serangan terhadap wilayah venezuela namun dia tidak menyebutkan lebih lanjut soal target dari serangan tersebut amerika serikat telah berhasil melancarkan serangan skala besar terhadap venezuela dan pemimpinnya presiden nicolas maduro yang bersama istrinya telah ditangkap dan diterbangkan ke luar negara tersebut kata trump dalam pernyataannya dia hanya mengatakan bahwa dirinya akan menggelar konferensi pers pada sabtu  siang waktu as di kediamannya maralago di florida operasi ini dilakukan bekerja sama dengan penegak hukum as detail lebih lanjut akan menyusul sebutnya akan ada konferensi pers hari ini pukul  waktu setempat di maralago terima kasih atas perhatian anda terhadap masalah ini tandas trump dalam pernyataannya"

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")

In [ ]:
teks_berita = "selamat datang di indonesia"

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")

In [ ]:
teks_berita = "sangat disayangkan namun kita sudah memberikan yang terbaik untuk bisa berjuang di ranah internasional"

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")

In [ ]:
teks_berita = "Futsal Indonesia tidak lagi dipandang sebelah mata di peta kekuatan dunia. Memasuki tahun 2026, Timnas Futsal Indonesia, yang dijuluki Skuad Garuda, telah menunjukkan transformasi luar biasa yang membawa mereka dari sekadar peserta menjadi penantang gelar yang disegani di level internasional."

hasil = classifier(teks_berita)


mapping = {"LABEL_0": "Neutral", "LABEL_1": "Positive", "LABEL_2": "Negative"}

print(f"\nKalimat: {teks_berita}")
print(f"Hasil: {mapping.get(hasil[0]['label'], hasil[0]['label'])} ({hasil[0]['score']:.4f})")